# ANTECIPA — Pipeline de dados reais do PNCP (via GitHub)

Projeto de dissertação (FACE/UnB): gestão preditiva de riscos em compras públicas no STF.

Esta versão do notebook **não precisa de acesso ao Google Drive**: ela clona o
repositório público, que já inclui os caches de coleta (`dados/brutos/`) —
qualquer pessoa consegue reproduzir o treino dos modelos em ~5 minutos, sem
refazer a coleta (~1 h) e sem credenciais.

> Resultados gravados aqui ficam só na sessão do Colab (que é descartável).
> Para persistir, baixe os arquivos gerados ou faça fork do repositório.

In [ ]:
# 1. Clonar o repositório (inclui código + caches de dados públicos)
!git clone --depth 1 https://github.com/DavidNadlerPrata/antecipa-pncp.git
%cd antecipa-pncp/pipeline

In [ ]:
# 2. Dependências (o Colab já traz scikit-learn; o comando só garante a versão)
%pip install -q scikit-learn

## 3. Coleta (opcional — normalmente pulada)

Os scripts são idempotentes: com os caches do repositório presentes, cada
etapa termina em segundos. Só recoleta se você apagar o JSON correspondente
em `dados/brutos/`. Coleta completa do zero: ~40–60 min, sujeita a
rate-limit do PNCP (HTTP 429; use `repara_multi.py` depois).

In [ ]:
!python coleta_pncp.py        # STF: contratos, termos, fornecedores, itens
!python coleta_precos_v2.py   # preços de referência (PDM/CATMAT)
!python coleta_multi.py       # 10 tribunais federais: contratos + termos
!python repara_multi.py       # refaz janelas que falharam por rate-limit
!python coleta_cadastros.py   # Receita Federal p/ fornecedores únicos

## 4. Datasets e treino (v1 e v2)

- **v1**: rótulo de desfecho adverso sem horizonte → `relatorio_modelo.md`
- **v2**: rótulo com horizonte fixo de 12 meses (corrige censura) →
  `relatorio_modelo_v2.md`
- Experimento: treino só-STF × multi-órgão no mesmo teste (com bootstrap)

In [ ]:
!python monta_dataset.py
!python treina_modelo.py
!python monta_dataset_v2.py
!python treina_modelo_v2.py
!python treina_stf_vs_multi.py

In [ ]:
# Ver o relatório v2 renderizado
from pathlib import Path
from IPython.display import Markdown, display
base = Path.cwd().parent / 'dados' / 'processados'
display(Markdown((base / 'relatorio_modelo_v2.md').read_text(encoding='utf-8')))

## 5. Publicar modelo calibrado e gerar o dashboard

In [ ]:
!python publica_modelo_v2.py
!python processa.py
!python gera_dashboard.py
# gera_dashboard grava o HTML dois níveis acima da pasta do projeto
import shutil
from pathlib import Path
destino = Path.cwd().parent / 'ANTECIPA_dashboard_real.html'
shutil.copy(Path.cwd().parent.parent / 'ANTECIPA_dashboard_real.html', destino)
print('Dashboard em:', destino)

In [ ]:
# Visualizar o dashboard no notebook e/ou baixar
from IPython.display import HTML, display
from pathlib import Path
html = (Path.cwd().parent / 'ANTECIPA_dashboard_real.html').read_text(encoding='utf-8')
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" width="100%" height="800"></iframe>'))

# Para baixar: descomente as duas linhas abaixo
# from google.colab import files
# files.download(str(Path.cwd().parent / 'ANTECIPA_dashboard_real.html'))

---
### Notas

- Fontes de dados: APIs públicas do PNCP, compras.gov.br e dados abertos do
  CNPJ (Receita Federal via minhareceita.org). Metodologia e limitações:
  `README.md` do repositório.
- A sessão do Colab é efêmera — nada é gravado no repositório. Para
  contribuir com mudanças, faça fork e abra um pull request.